# KMX MTN-Specific RAGU Score Notebook
This notebook computes RAGU scores for KMX loans, broken down by MTN model (3.0, 3.1, 3.2, 4.1).
Aligned with `bareboned_ragu_new.ipynb` for diagnostic purposes.
- **Granularity:** Configurable (weekly/monthly/quarterly) via `granularity` in cell 1
- **Date handling:** Weekly uses `app_date`, monthly/quarterly use `book_date`
- **Output:** Per-model RAGU decomposition + diagnostics, exported to `new_kmx_models.xlsx`

## How to Run
1. Set `granularity`, `START_DATE`, `END_DATE` in cell 1
2. Set `run_from_pickle = True` to load pre-computed data (faster), or `False` to re-run SQL queries
3. Run All Cells
4. Results are exported to `new_kmx_models.xlsx`

In [1]:
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os

# ── Configuration ──
granularity = 'w'                # 'q' = quarterly, 'm' = monthly, 'w' = weekly

START_DATE = '2025-12-01'
END_DATE = None                  # None = auto-detect from today's date

run_every_query = False           # True = re-run SQL, False = load pickle
run_from_pickle = not run_every_query

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}
PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}

BASELINES = {
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'kmx_loss_scale': 0.067,
}

MTN_MODELS = [3.0, 3.1, 3.2, 4.1]
EXCEL_OUTPUT = 'new_kmx_models.xlsx'

# ── Derived values ──
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()
period_freq = PERIOD_FREQ_MAP[granularity]
start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)
min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: w
Date column: app_date
Period range: 2025-11-30/2025-12-06 to 2026-05-03/2026-05-09
SQL min_date: '2025-12-01'


In [2]:
# Parameters
granularity = "w"
START_DATE = "2026-01-01"
END_DATE = None
run_every_query = False
BASELINES = {"KMX": {"ltv": 1.59, "new_recovery_unadjusted": 0.58, "apr": 0.25}}
MODEL_PARAMS = {"mean_unit_loss": 0.5, "unit_loss_to_model_score": 0.02, "kmx_loss_scale": 0.067}
MTN_MODELS = [3.0, 3.1, 3.2, 4.1]


In [3]:
def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings."""
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


def rebuild_ms_df(ula_subset):
    """Recompute ms_df from a filtered ula_df subset (for per-model scoring).
    MTN 4.1 transform is already applied in-place on ula_df_total upstream."""
    ms_source = ula_subset[['lob', 'cd_model_score', 'amt_financed', 'period']].copy()
    ms_source = ms_source.rename(columns={'cd_model_score': 'model_score'})
    ms = ms_source.groupby(['period', 'lob']).apply(
        weighted_average_and_sum, 'model_score', include_groups=False
    ).reset_index()
    ms['period'] = format_vintage(ms['period'])
    return ms

In [4]:
# Per-table caches under cache/ with _v1 schema tags. When run_from_pickle=True
# (i.e. run_every_query=False) each cached_sql call reuses its pickle if present
# and falls through to SQL if missing. Delete an individual pickle to force a
# selective refresh.
#
# Note: ULA is cached under cache/ula_kmx_v1.pkl because this notebook
# applies a KMX-only filter via sub_list; that is a different schema (KMX
# subset of rows) than the full-LOB cache/ula_v1.pkl written by
# bareboned_ragu_new.ipynb. DLA and new_recovery use identical queries in both
# notebooks, so they share the same cache files.

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('cache/ula_kmx_v1.pkl', 'cache/dla_v1.pkl', 'cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            'vintage_level_ula_query.txt', 'cache/ula_kmx_v1.pkl',
            sub_list=[('{min_book_date}', f"{min_date_sql}\n  AND dru.riskdealergroup = 'KMX'")],
            connection=conn, force_refresh=force,
        )
        print('ULA ready')

        dla_df = cached_sql(
            'new_dll_query.txt', 'cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            'new_recovery_queryt.txt', 'cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('cache/ula_kmx_v1.pkl')
    dla_df = get_pickle('cache/dla_v1.pkl')
    new_recovery = get_pickle('cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


ULA, DLA, New recovery loaded from cache
ULA records: 195,304
[PROGRESS] Data Fetch Complete


In [5]:
def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']
    ula_df['loss_multiplier'] = 1.0

    if leave_out!='Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)

    if leave_out!='Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)

    if leave_out!='High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag \
                                    + 0.05 * ula_df.high_pti_tier_1_flag \
                                    + 0.1 * ula_df.high_pti_tier_2_flag \
                                    + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag

    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] /= (1 + loss_scale)

    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)\
                                        * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)

    if leave_out!='Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out!='Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out!='Clip':
        clipped_multiplier = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
        ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped_multiplier

    if leave_out!='Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier']  *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out !='npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 +  0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]),'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out!='Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier']  *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out!='georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier']  *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out!='txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140) ,'loss_multiplier']  *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) &  (ula_df.cd_model_score >= 135)   ,'loss_multiplier']  *= 1 - 0.05 * ula_df.txca_flag

    if leave_out!='state_counter_adj':
        ula_df.loc[ ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1])  & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag) ,'loss_multiplier'] *= 1.012

    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag  - 0.18*ula_df.chime_flag* ula_df.soft_pull_flag) + 0.46*ula_df.chime_flag

    if leave_out!='Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out!='Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out!='Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out!='Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out!='Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag,'loss_multiplier'] *= 1.1  * (0.99 + 0.11*ula_df.low_bureau_flag)  * (0.978 + 0.172*ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag,'loss_multiplier'] *= (1 * (0.98 + 0.22*ula_df.low_bureau_flag)  * (0.945 + 0.405*ula_df.open_tl_flag ) /np.maximum(ula_df.cd_perc_flag*ula_df.open_tl_flag*1.2,1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) &  ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97  + 0.15  * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1)         &  ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0   + 0.05  * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1)         & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0   + 0.10  * ula_df.cd_perc_flag


    if leave_out!='blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1


    if leave_out!='Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df


def get_ula_multiplier_kmx_diag(ula_df, loss_scale=None, leave_out='None', verbose=True):
    """Identical logic to get_ula_multiplier_kmx, but also returns step-by-step loss_multiplier means and flag means."""
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    if verbose:
        print(f"00_initial  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag) \
                                    + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)
    _record('02_low_fico_3.0', ula_df)
    if verbose:
        print(f"02_low_fico | flag mean: {ula_df.low_fico_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag) \
                                    + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)
    _record('03_low_vantage_3.0', ula_df)
    if verbose:
        print(f"03_low_vant | flag mean: {ula_df.low_vantage_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag \
                                    + 0.05 * ula_df.high_pti_tier_1_flag \
                                    + 0.1 * ula_df.high_pti_tier_2_flag \
                                    + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag
    _record('04_high_pti_3.0', ula_df)
    if verbose:
        print(f"04_high_pti | tier1: {ula_df.high_pti_tier_1_flag.mean():.6f}  tier2: {ula_df.high_pti_tier_2_flag.mean():.6f}  tier3: {ula_df.high_pti_tier_3_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)
    _record('07_loss_scale_div_3.0', ula_df)
    if verbose:
        print(f"07_scale_dv | dividing by (1 + {loss_scale})  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag) \
                                        * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)
    _record('08_secured_credit_3.0', ula_df)
    if verbose:
        print(f"08_sec_cred | flag mean: {ula_df.secured_credit_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    _record('09_auth_tradelines_3.0', ula_df)
    if verbose:
        print(f"09_auth_tl  | flag mean: {ula_df.kmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))
    _record('11_soft_pull_3.0', ula_df)
    if verbose:
        print(f"11_soft_pll | flag mean: {ula_df.soft_pull_flag.mean():.6f}  narrowed: {ula_df.narrowed_soft_pull_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    _record('12_fraud_all', ula_df)
    if verbose:
        print(f"12_fraud    | fraud_adj mean: {ula_df.fraud_adjustment.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        clipped_multiplier = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
        ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped_multiplier
    _record('13_clip_3.0', ula_df)
    if verbose:
        print(f"13_clip_3.0 | clip [0.8, {1.35 / (1 + loss_scale):.4f}]  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    _record('14_vehicle_age_3.0', ula_df)
    if verbose:
        print(f"14_veh_age  | continuous_age mean: {ula_df.continuous_vehicle_age.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    _record('15_npc_all', ula_df)
    if verbose:
        print(f"15_npc      | npc_flag mean: {ula_df.kmx_npc_flag.mean():.6f}  high_pti_npc mean: {ula_df.high_pti_npc.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    _record('16_student_loans_all', ula_df)
    if verbose:
        print(f"16_stu_loan | flag mean: {ula_df.student_loan_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    _record('17_high_sales_price_4.1', ula_df)
    if verbose:
        print(f"17_hi_price | flag mean: {ula_df.high_sales_price_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    _record('18_driver_flag_all', ula_df)
    if verbose:
        print(f"18_driver   | flag mean: {ula_df.driver_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    _record('19_louisiana_all', ula_df)
    if verbose:
        print(f"19_louisiana| flag mean: {ula_df.louisiana_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    _record('20_georgia_all', ula_df)
    if verbose:
        print(f"20_georgia  | flag mean: {ula_df.georgia_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    _record('21_txca_all', ula_df)
    if verbose:
        print(f"21_txca     | flag mean: {ula_df.txca_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'state_counter_adj':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag), 'loss_multiplier'] *= 1.012
    _record('22_state_counter_all', ula_df)
    if verbose:
        no_state_adj = ((~ula_df.louisiana_flag) & (~ula_df.georgia_flag) & (~ula_df.txca_flag)).mean()
        print(f"22_st_cntr  | no state adj: {no_state_adj:.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag
    _record('23_secured_credit_3.1+', ula_df)
    if verbose:
        print(f"23_sec_cr31 | chime mean: {ula_df.chime_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    _record('24_job_time_3.1+', ula_df)
    if verbose:
        print(f"24_job_t_31 | flag mean: {ula_df.job_time_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    _record('25_existing_dq_3.1+', ula_df)
    if verbose:
        print(f"25_dq_31    | flag mean: {ula_df.existing_dq_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    _record('26_employment_3.1+', ula_df)
    if verbose:
        print(f"26_emp_31   | seasonal: {ula_df.seasonal_employment_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    _record('27_auth_tradelines_3.1+', ula_df)
    if verbose:
        print(f"27_auth_31  | flag mean: {ula_df.kmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    _record('28_soft_pull_3.1+', ula_df)
    if verbose:
        mtn31_mask = ula_df.mtn_model.isin([3.1, 3.2, 4.1])
        print(f"28_soft_31  | soft_pull: {ula_df.soft_pull_flag.mean():.6f}  low_bureau: {ula_df.low_bureau_flag.mean():.6f}  cd_perc: {ula_df.cd_perc_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1
    _record('28b_blanket_3.1+', ula_df)
    if verbose:
        print(f"28b_blanket | MTN 3.1+ /= 1.1  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)
    _record('29_final_clip_3.1+', ula_df)
    if verbose:
        print(f"29_final_31 | clip [0.75, 1.4] for MTN 3.1+  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")
        print("\n--- Per MTN Model Means ---")
        for model in [3.0, 3.1, 3.2, 4.1]:
            mean_val = ula_df.loc[ula_df.mtn_model == model, 'loss_multiplier'].mean()
            count_val = (ula_df.mtn_model == model).sum()
            print(f"mtn_model {model}: mean loss_multiplier = {mean_val:.6f}  (n={count_val})")

    n = len(ula_df)
    flags = {
        'job_time_flag': ula_df.job_time_flag.mean(),
        'low_fico_flag': ula_df.low_fico_flag.mean(),
        'high_model_score_flag': ula_df.high_model_score_flag.mean(),
        'low_vantage_flag': ula_df.low_vantage_flag.mean(),
        'normal_pti_flag': ula_df.normal_pti_flag.mean(),
        'high_pti_tier_1_flag': ula_df.high_pti_tier_1_flag.mean(),
        'high_pti_tier_2_flag': ula_df.high_pti_tier_2_flag.mean(),
        'high_pti_tier_3_flag': ula_df.high_pti_tier_3_flag.mean(),
        'existing_dq_flag': ula_df.existing_dq_flag.mean(),
        'seasonal_employment_flag': ula_df.seasonal_employment_flag.mean(),
        'secured_credit_flag': ula_df.secured_credit_flag.mean(),
        'kmx_auth_tradelines_flag': ula_df.kmx_auth_tradelines_flag.mean(),
        'null_fico_w_vantage_flag': ula_df.null_fico_w_vantage_flag.mean(),
        'null_fico_null_vantage_flag': ula_df.null_fico_null_vantage_flag.mean(),
        'soft_pull_flag': ula_df.soft_pull_flag.mean(),
        'narrowed_soft_pull_flag': ula_df.narrowed_soft_pull_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'continuous_vehicle_age_mean': ula_df.continuous_vehicle_age.mean(),
        'kmx_npc_flag': ula_df.kmx_npc_flag.mean(),
        'high_pti_npc': ula_df.high_pti_npc.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'high_sales_price_flag': ula_df.high_sales_price_flag.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'louisiana_flag': ula_df.louisiana_flag.mean(),
        'georgia_flag': ula_df.georgia_flag.mean(),
        'txca_flag': ula_df.txca_flag.mean(),
        'chime_flag': ula_df.chime_flag.mean(),
        'low_bureau_flag': ula_df.low_bureau_flag.mean(),
        'cd_perc_flag': ula_df.cd_perc_flag.mean(),
        'open_tl_flag': ula_df.open_tl_flag.mean(),
        'mtn_3_1_flag': ula_df.mtn_3_1_flag.mean(),
        'mtn_model_3.0_pct': (ula_df.mtn_model == 3.0).mean(),
        'mtn_model_3.1_pct': (ula_df.mtn_model == 3.1).mean(),
        'mtn_model_3.2_pct': (ula_df.mtn_model == 3.2).mean(),
        'mtn_model_4.1_pct': (ula_df.mtn_model == 4.1).mean(),
    }

    return ula_df, pd.Series(steps), pd.Series(flags)

In [6]:
mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None'):
    """Core RAGU Score calculation for a single vintage and individual LOB."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
    apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df, leave_out=leave_out)
    else:
        raise ValueError("This notebook only supports KMX")

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)

    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df

In [7]:
# Filter out Core LOB
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns using pd.to_period
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')
    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select([ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
                                      ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
                                     ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# Driver flag
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# Handle NA values
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# Weekly-matching filters
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

# KMX Flags
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 475))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0

# Deduplicate driver flags
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# Vintage assignment
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# Filter to KMX only
ula_df_total = ula_df_total[ula_df_total.lob == 'KMX'].copy()
new_recovery = new_recovery[new_recovery.lob == 'KMX'].copy()

# MTN 4.1 model score transformation (applied in-place to ULA source)
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

# Aggregate model scores from ULA data (same source as ltv/apr)
ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f'Data loaded: {len(ula_df_total)} ULA rows')
print(f'MTN model distribution:\n{ula_df_total.mtn_model.value_counts().sort_index()}')
print(f'ms_df: {len(ms_df)} rows')
print(f'Available KMX vintages: {sorted(ula_df_total.vintage.unique())}')

Data loaded: 31654 ULA rows
MTN model distribution:
mtn_model
3.0      246
3.1    17503
3.2    10682
4.1     3223
Name: count, dtype: int64
ms_df: 21 rows
Available KMX vintages: ['2025-11-30/2025-12-06', '2025-12-07/2025-12-13', '2025-12-14/2025-12-20', '2025-12-21/2025-12-27', '2025-12-28/2026-01-03', '2026-01-04/2026-01-10', '2026-01-11/2026-01-17', '2026-01-18/2026-01-24', '2026-01-25/2026-01-31', '2026-02-01/2026-02-07', '2026-02-08/2026-02-14', '2026-02-15/2026-02-21', '2026-02-22/2026-02-28', '2026-03-01/2026-03-07', '2026-03-08/2026-03-14', '2026-03-15/2026-03-21', '2026-03-22/2026-03-28', '2026-03-29/2026-04-04', '2026-04-05/2026-04-11', '2026-04-12/2026-04-18', '2026-04-19/2026-04-25']


In [8]:
baseline_config = BASELINES['KMX']

results_by_model = {}

all_vintages = sorted(ula_df_total['vintage'].unique())

for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    print(f'\n{"="*60}')
    print(f'  Processing: {label}')
    print(f'{"="*60}')

    # Filter DataFrames by MTN model
    if mtn_model_filter == 'All KMX':
        ula_filtered = ula_df_total.copy()
        new_rec_filtered = new_recovery.copy()
        ms_filtered = ms_df.copy()
    else:
        mtn_accounts = set(ula_df_total[ula_df_total.mtn_model == mtn_model_filter].account_number)
        ula_filtered = ula_df_total[ula_df_total.mtn_model == mtn_model_filter].copy()
        new_rec_filtered = new_recovery[new_recovery.account_number.isin(mtn_accounts)].copy()
        ms_filtered = rebuild_ms_df(ula_filtered)

    n_loans = len(ula_filtered)
    if n_loans == 0:
        print(f'  No loans found for {label}, skipping.')
        results_by_model[mtn_model_filter] = pd.DataFrame()
        continue

    print(f'  Loans: {n_loans}')

    full_df_list = []
    vintages_processed = []

    for vintage in all_vintages:
        n_vintage = len(ula_filtered[ula_filtered.vintage == vintage])
        if n_vintage == 0:
            continue

        print(f'  {vintage} KMX ({label}, n={n_vintage})')
        full_df = get_ragu_score(vintage, 'KMX', ula_filtered, new_rec_filtered, ms_filtered, baseline_config)
        if full_df is not None:
            full_df_list.append(full_df)
            vintages_processed.append(vintage)

    if full_df_list:
        all_df = pd.concat(full_df_list)
        results_by_model[mtn_model_filter] = all_df
        print(f'\n  {label}: {len(vintages_processed)} vintages processed')
    else:
        results_by_model[mtn_model_filter] = pd.DataFrame()
        print(f'\n  {label}: No vintages had data')

print(f'\n{"="*60}')
print('  All models processed.')
print(f'{"="*60}')
print("[PROGRESS] Scoring Complete")



  Processing: MTN 3.0
  Loans: 246
  2025-11-30/2025-12-06 KMX (MTN 3.0, n=246)



  MTN 3.0: 1 vintages processed

  Processing: MTN 3.1


  Loans: 17503
  2025-11-30/2025-12-06 KMX (MTN 3.1, n=674)
  2025-12-07/2025-12-13 KMX (MTN 3.1, n=965)


  2025-12-14/2025-12-20 KMX (MTN 3.1, n=1141)


  2025-12-21/2025-12-27 KMX (MTN 3.1, n=790)
  2025-12-28/2026-01-03 KMX (MTN 3.1, n=1060)
  2026-01-04/2026-01-10 KMX (MTN 3.1, n=1124)


...

  2025-12-21/2025-12-27 KMX (All KMX, n=915)
  2025-12-28/2026-01-03 KMX (All KMX, n=1202)
  2026-01-04/2026-01-10 KMX (All KMX, n=1291)
  2026-01-11/2026-01-17 KMX (All KMX, n=1320)


  2026-01-18/2026-01-24 KMX (All KMX, n=1279)
  2026-01-25/2026-01-31 KMX (All KMX, n=1132)
  2026-02-01/2026-02-07 KMX (All KMX, n=1398)
  2026-02-08/2026-02-14 KMX (All KMX, n=1427)


  2026-02-15/2026-02-21 KMX (All KMX, n=2038)
  2026-02-22/2026-02-28 KMX (All KMX, n=3409)
  2026-03-01/2026-03-07 KMX (All KMX, n=2599)
  2026-03-08/2026-03-14 KMX (All KMX, n=2442)


  2026-03-15/2026-03-21 KMX (All KMX, n=1876)
  2026-03-22/2026-03-28 KMX (All KMX, n=1767)
  2026-03-29/2026-04-04 KMX (All KMX, n=1635)
  2026-04-05/2026-04-11 KMX (All KMX, n=1291)


  2026-04-12/2026-04-18 KMX (All KMX, n=1221)
  2026-04-19/2026-04-25 KMX (All KMX, n=312)

  All KMX: 21 vintages processed

  All models processed.
[PROGRESS] Scoring Complete


In [9]:
"""
Diagnostics: Run get_ula_multiplier_kmx_diag for each MTN model and for All KMX combined.
Produces step-by-step loss_multiplier traces and flag means per vintage.
"""

def build_output_df(results, wtd_mults, record_counts, ragu_gli_dict=None):
    """Helper to build multiplier steps and summary DataFrame."""
    if not results:
        return None
    step_names = list(next(iter(results.values())).keys())
    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        col_data[col_name] = [step_dict.get(s) for s in step_names]
    diag_df = pd.DataFrame(col_data, index=step_names)
    summary = {}
    for (lob, vintage) in results.keys():
        col = f"{lob} | {vintage}"
        final_mult_mean = diag_df[col].iloc[-1]
        wtd_mult = wtd_mults.get((lob, vintage), float('nan'))
        if ragu_gli_dict is not None:
            gross_loss_impact = ragu_gli_dict.get((lob, vintage), 25 * (1 - wtd_mult))
        else:
            gross_loss_impact = 25 * (1 - wtd_mult)
        summary[col] = {
            '--- FINAL_MULT (mean)': final_mult_mean,
            '--- WTD_MULT_RAGU': wtd_mult,
            '--- GROSS_LOSS_IMPACT': gross_loss_impact,
            '--- N_RECORDS': record_counts.get((lob, vintage), 0),
        }
    summary_df = pd.DataFrame(summary)
    return pd.concat([diag_df, summary_df])

def build_flags_df(flag_results, record_counts):
    """Helper to build flag means DataFrame."""
    if not flag_results:
        return None
    flag_names = list(next(iter(flag_results.values())).keys())
    flag_col_data = {}
    for (lob, vintage), flag_dict in flag_results.items():
        col_name = f"{lob} | {vintage}"
        flag_col_data[col_name] = [flag_dict.get(f) for f in flag_names]
    flags_df = pd.DataFrame(flag_col_data, index=flag_names)
    n_row = {}
    for (lob, vintage) in flag_results.keys():
        n_row[f"{lob} | {vintage}"] = record_counts.get((lob, vintage), 0)
    flags_df.loc['--- N_RECORDS'] = n_row
    return flags_df


STEP_LABEL_MAP = {
    '00_initial':              'Initial (1.0)',
    '02_low_fico_3.0':         'Low FICO (3.0)',
    '03_low_vantage_3.0':      'Low Vantage (3.0)',
    '04_high_pti_3.0':         'High PTI (3.0)',
    '07_loss_scale_div_3.0':   'Loss Scale Div (3.0)',
    '08_secured_credit_3.0':   'Secured Credit (3.0)',
    '09_auth_tradelines_3.0':  'Auth Tradelines (3.0)',
    '11_soft_pull_3.0':        'Soft Pull (3.0)',
    '12_fraud_all':            'Fraud Adjustment',
    '13_clip_3.0':             'Clip (3.0)',
    '14_vehicle_age_3.0':      'Vehicle Age (3.0)',
    '15_npc_all':              'NPC',
    '16_student_loans_all':    'Student Loans',
    '17_high_sales_price_4.1': 'High Sales Price (4.1)',
    '18_driver_flag_all':      'Driver Flag',
    '19_louisiana_all':        'Louisiana',
    '20_georgia_all':          'Georgia',
    '21_txca_all':             'TX / CA',
    '22_state_counter_all':    'State Counter',
    '23_secured_credit_3.1+':  'Secured Credit / Chime (3.1+)',
    '24_job_time_3.1+':        'Job Time (3.1+)',
    '25_existing_dq_3.1+':     'Existing DQ (3.1+)',
    '26_employment_3.1+':      'Employment Type (3.1+)',
    '27_auth_tradelines_3.1+': 'Auth Tradelines (3.1+)',
    '28_soft_pull_3.1+':       'Soft Pull (3.1+)',
    '28b_blanket_3.1+':        'Blanket Adjustment (3.1+)',
    '29_final_clip_3.1+':      'Clip (3.1+)',
}


def build_attribution_df(results, ragu_gli_dict):
    """
    Decompose gross_loss_impact across multiplier steps using logarithmic attribution.

    For each vintage:
      1. Compute per-step ratios: r_i = step_i / step_{i-1}
      2. Log shares: ln(r_i) / ln(final_multiplier)  [proportional contribution]
      3. Attributed impact: share_i * gross_loss_impact

    gross_loss_impact is sourced from get_ragu_score output (results_by_model) to ensure
    the attribution decomposes the same value that appears in the RAGU score decomposition.

    When the final multiplier equals 1.0 (no net adjustment), all impacts are 0.
    """
    if not results:
        return None

    step_keys = list(next(iter(results.values())).keys())
    adjustment_keys = [k for k in step_keys if k != '00_initial']
    readable_labels = [STEP_LABEL_MAP.get(k, k) for k in adjustment_keys]

    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        gross_loss_impact = ragu_gli_dict.get((lob, vintage), float('nan'))

        cumulative = [step_dict.get(k, float('nan')) for k in step_keys]
        ratios = []
        for i, k in enumerate(step_keys):
            if k == '00_initial':
                continue
            prev = cumulative[i - 1]
            curr = cumulative[i]
            if prev and prev != 0:
                ratios.append(curr / prev)
            else:
                ratios.append(1.0)

        import math
        final_mult = cumulative[-1]
        log_final = math.log(final_mult) if final_mult and final_mult > 0 and abs(final_mult - 1.0) > 1e-12 else None

        if log_final is None or pd.isna(gross_loss_impact):
            col_data[col_name] = [0.0] * len(adjustment_keys) + [gross_loss_impact if not pd.isna(gross_loss_impact) else 0.0]
        else:
            log_ratios = [math.log(r) if r and r > 0 else 0.0 for r in ratios]
            attributed = [(lr / log_final) * gross_loss_impact for lr in log_ratios]
            col_data[col_name] = attributed + [sum(attributed)]

    index_labels = readable_labels + ['--- TOTAL (check)']
    return pd.DataFrame(col_data, index=index_labels)

diag_results_by_model = {}

for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    
    if mtn_model_filter == 'All KMX':
        ula_diag_source = ula_df_total.copy()
    else:
        ula_diag_source = ula_df_total[ula_df_total.mtn_model == mtn_model_filter].copy()
    
    if len(ula_diag_source) == 0:
        print(f'\n{label}: No data for diagnostics, skipping.')
        diag_results_by_model[mtn_model_filter] = (None, None, None)
        continue
    
    # Build ragu_gli_dict from results_by_model for this MTN model
    ragu_gli_dict = {}
    model_df = results_by_model.get(mtn_model_filter)
    if model_df is not None and len(model_df) > 0:
        for _, row in model_df.reset_index().iterrows():
            ragu_gli_dict[('KMX', row['vintage'])] = row['gross_loss_impact']
    
    # Get the most recent vintages for diagnostics
    target_vintages = sorted(ula_diag_source.vintage.unique())[-6:]
    
    kmx_results = {}
    kmx_flag_results = {}
    kmx_wtd_mults = {}
    kmx_record_counts = {}
    
    print(f'\n{"="*60}')
    print(f'  DIAGNOSTICS: {label}')
    print(f'{"="*60}')
    
    for vintage in target_vintages:
        ula_vintage = ula_diag_source[ula_diag_source.vintage == vintage].copy()
        n = len(ula_vintage)
        if n == 0:
            continue
        
        print(f'\n{"="*60}')
        print(f'  KMX {vintage} ({label})  (n={n})')
        print(f'{"="*60}')
        
        ula_vintage_diag, steps, flags = get_ula_multiplier_kmx_diag(ula_vintage, loss_scale=MODEL_PARAMS['kmx_loss_scale'], leave_out='None', verbose=True)
        
        diag_mix = ula_vintage_diag[['account_number', 'bbvalue', 'amt_financed', 'loss_multiplier']].copy()
        nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
            subset='account_number', keep='first')
        diag_mix = diag_mix.merge(nr, on='account_number', how='left').drop_duplicates(
            subset='account_number', keep='first')
        bb_pop = diag_mix[diag_mix['bbvalue'].notna() & (diag_mix['bbvalue'] > 0)]
        if len(bb_pop) > 0 and bb_pop.amt_financed.sum() > 0:
            wtd_mult = (bb_pop.loss_multiplier * bb_pop.amt_financed).sum() / bb_pop.amt_financed.sum()
        else:
            wtd_mult = float('nan')
        
        ragu_gli = ragu_gli_dict.get(('KMX', vintage), float('nan'))
        print(f"  bb_populated: {len(bb_pop)} / {n}  wtd_mult: {wtd_mult:.6f}  ragu_gli: {ragu_gli:.4f}")
        
        kmx_results[('KMX', vintage)] = steps.to_dict()
        kmx_flag_results[('KMX', vintage)] = flags.to_dict()
        kmx_wtd_mults[('KMX', vintage)] = wtd_mult
        kmx_record_counts[('KMX', vintage)] = n
    
    kmx_output_df = build_output_df(kmx_results, kmx_wtd_mults, kmx_record_counts, ragu_gli_dict)
    kmx_flags_df = build_flags_df(kmx_flag_results, kmx_record_counts)
    kmx_attribution_df = build_attribution_df(kmx_results, ragu_gli_dict)
    diag_results_by_model[mtn_model_filter] = (kmx_output_df, kmx_flags_df, kmx_attribution_df)

    if kmx_output_df is not None:
        print(f'\n--- {label} Multiplier Steps ---')
        display(kmx_output_df)
        print(f'\n--- {label} Flag Means ---')
        display(kmx_flags_df)
    if kmx_attribution_df is not None:
        print(f'\n--- {label} Gross Loss Attribution ---')
        display(kmx_attribution_df)


  DIAGNOSTICS: MTN 3.0

  KMX 2025-11-30/2025-12-06 (MTN 3.0)  (n=246)
00_initial  | loss_multiplier mean: 1.000000
02_low_fico | flag mean: 0.000000  | loss_multiplier mean: 1.000000
03_low_vant | flag mean: 0.012195  | loss_multiplier mean: 1.002305
04_high_pti | tier1: 0.109756  tier2: 0.036585  tier3: 0.000000  | loss_multiplier mean: 1.011451
07_scale_dv | dividing by (1 + 0.067)  | loss_multiplier mean: 0.947939
08_sec_cred | flag mean: 0.382114  | loss_multiplier mean: 0.958046
09_auth_tl  | flag mean: 0.036585  | loss_multiplier mean: 0.954776
11_soft_pll | flag mean: 0.849593  narrowed: 0.142276  | loss_multiplier mean: 0.932097
12_fraud    | fraud_adj mean: 1.009024  | loss_multiplier mean: 0.940293
13_clip_3.0 | clip [0.8, 1.2652]  | loss_multiplier mean: 0.938785
14_veh_age  | continuous_age mean: 5.513550  | loss_multiplier mean: 0.910912
15_npc      | npc_flag mean: 0.247967  high_pti_npc mean: 0.146341  | loss_multiplier mean: 0.912195
16_stu_loan | flag mean: 0.243902 

,KMX | 2025-11-30/2025-12-06
00_initial,1.000000
02_low_fico_3.0,1.000000
03_low_vantage_3.0,1.002305
04_high_pti_3.0,1.011451
07_loss_scale_div_3.0,0.947939
08_secured_credit_3.0,0.958046
09_auth_tradelines_3.0,0.954776
11_soft_pull_3.0,0.932097
12_fraud_all,0.940293
13_clip_3.0,0.938785



--- MTN 3.0 Flag Means ---


,KMX | 2025-11-30/2025-12-06
job_time_flag,0.178862
low_fico_flag,0.000000
high_model_score_flag,0.227642
low_vantage_flag,0.012195
normal_pti_flag,0.853659
high_pti_tier_1_flag,0.109756
high_pti_tier_2_flag,0.036585
high_pti_tier_3_flag,0.000000
existing_dq_flag,0.117886
seasonal_employment_flag,0.012195



--- MTN 3.0 Gross Loss Attribution ---


...

,KMX | 2026-03-15/2026-03-21,KMX | 2026-03-22/2026-03-28,KMX | 2026-03-29/2026-04-04,KMX | 2026-04-05/2026-04-11,KMX | 2026-04-12/2026-04-18,KMX | 2026-04-19/2026-04-25
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
02_low_fico_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
03_low_vantage_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
04_high_pti_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
07_loss_scale_div_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
08_secured_credit_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
09_auth_tradelines_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
11_soft_pull_3.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
12_fraud_all,0.992761,1.001064,1.011303,1.015484,1.009058,1.000000
13_clip_3.0,0.992761,1.001064,1.011303,1.015484,1.009058,1.000000



--- All KMX Flag Means ---


,KMX | 2026-03-15/2026-03-21,KMX | 2026-03-22/2026-03-28,KMX | 2026-03-29/2026-04-04,KMX | 2026-04-05/2026-04-11,KMX | 2026-04-12/2026-04-18,KMX | 2026-04-19/2026-04-25
job_time_flag,0.113539,0.119977,0.098471,0.105345,0.089271,0.108974
low_fico_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
high_model_score_flag,0.314499,0.327674,0.334557,0.317583,0.315315,0.362179
low_vantage_flag,0.023454,0.015846,0.020795,0.010070,0.011466,0.006410
normal_pti_flag,0.905650,0.891907,0.893578,0.900077,0.900082,0.884615
high_pti_tier_1_flag,0.059168,0.072439,0.074006,0.062742,0.067977,0.080128
high_pti_tier_2_flag,0.030917,0.033390,0.027523,0.031758,0.030303,0.035256
high_pti_tier_3_flag,0.004264,0.002264,0.004893,0.005422,0.001638,0.000000
existing_dq_flag,0.131130,0.118846,0.111315,0.108443,0.142506,0.134615
seasonal_employment_flag,0.007996,0.008489,0.006116,0.009295,0.007371,0.012821



--- All KMX Gross Loss Attribution ---


,KMX | 2026-03-15/2026-03-21,KMX | 2026-03-22/2026-03-28,KMX | 2026-03-29/2026-04-04,KMX | 2026-04-05/2026-04-11,KMX | 2026-04-12/2026-04-18,KMX | 2026-04-19/2026-04-25
Low FICO (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Low Vantage (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
High PTI (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Loss Scale Div (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Secured Credit (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Auth Tradelines (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Soft Pull (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Fraud Adjustment,-0.199532,0.000370,0.215263,1.843466,2.645161,-0.000000
Clip (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
Vehicle Age (3.0),0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000


In [10]:
def build_ragu_decomposition(all_df_model, original_model_scores_df, lob='KMX'):
    '''Build RAGU decomposition table from model results.'''
    if all_df_model is None or len(all_df_model) == 0:
        return None

    raw = all_df_model.loc[[lob]][['vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact',
                                   'ltv_impact', 'apr_impact', 'ragu_score', 'ltv']].reset_index(drop=True).set_index('vintage').T.copy()

    ms_for_lob = original_model_scores_df[original_model_scores_df.lob == lob].copy()
    ms_for_lob = ms_for_lob[ms_for_lob['period'].isin(raw.columns)]
    if len(ms_for_lob) > 0:
        raw = pd.concat([raw, ms_for_lob.drop(columns=['lob', 'amt_financed']).rename(columns={'period': 'vintage'}).set_index('vintage').T.rename({'model_score': 'contract_model_score'})])

    raw.index = ['Mountain3 Score', 'Gross Loss Impact', 'Recovery Impact',
                 'LTV Impact', 'APR Impact', 'RAGU Score', 'LTV', 'Contract Model Score']

    decomp = pd.DataFrame(index=['Contract Model Score', 'Mountain3 Score',
                                  'Gross Loss Adjustments', 'Recovery Adjustments',
                                  'LTV Adjustments', 'APR Adjustments',
                                  'RAGU Score', 'LTV'],
                           columns=raw.columns)

    decomp.loc['Contract Model Score'] = raw.loc['Contract Model Score']
    decomp.loc['Mountain3 Score'] = raw.loc['Mountain3 Score'] - raw.loc['Contract Model Score']
    decomp.loc['Gross Loss Adjustments'] = raw.loc['Gross Loss Impact']
    decomp.loc['Recovery Adjustments'] = raw.loc['Recovery Impact']
    decomp.loc['LTV Adjustments'] = raw.loc['LTV Impact']
    decomp.loc['APR Adjustments'] = raw.loc['APR Impact']
    decomp.loc['RAGU Score'] = raw.loc['RAGU Score']
    decomp.loc['LTV'] = raw.loc['LTV']

    return decomp

# Build original_model_scores for each MTN model subset
original_model_scores_by_model = {}
for mtn_model_filter in MTN_MODELS + ['All KMX']:
    if mtn_model_filter == 'All KMX':
        original_model_scores_by_model['All KMX'] = ms_df
    else:
        ula_subset = ula_df_total[ula_df_total.mtn_model == mtn_model_filter]
        if len(ula_subset) > 0:
            original_model_scores_by_model[mtn_model_filter] = rebuild_ms_df(ula_subset)
        else:
            original_model_scores_by_model[mtn_model_filter] = pd.DataFrame(columns=['period', 'lob', 'model_score', 'amt_financed'])

# Write to Excel
with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    for mtn_model_filter in ['All KMX'] + MTN_MODELS:
        label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
        all_df_model = results_by_model.get(mtn_model_filter)
        orig_ms = original_model_scores_by_model.get(mtn_model_filter)

        if all_df_model is None or len(all_df_model) == 0:
            print(f'{label}: No results to export, writing empty sheet.')
            pd.DataFrame({'Note': [f'No loans found for {label}']}).to_excel(writer, sheet_name=label, index=False)
            continue

        xlsx_df = build_ragu_decomposition(all_df_model, orig_ms, lob='KMX')
        if xlsx_df is not None:
            xlsx_df.to_excel(writer, sheet_name=label)
            print(f'\n{label} RAGU Decomposition:')
            display(xlsx_df)
        else:
            pd.DataFrame({'Note': [f'Could not build decomposition for {label}']}).to_excel(writer, sheet_name=label, index=False)

    # Diagnostics sheets
    for mtn_model_filter in ['All KMX'] + MTN_MODELS:
        label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
        diag_data = diag_results_by_model.get(mtn_model_filter, (None, None, None))
        output_df, flags_df, attribution_df = diag_data

        sheet_mult = f'{label} Mult Steps'[:31]
        sheet_flags = f'{label} Flags'[:31]
        sheet_attr = f'{label} Attribution'[:31]

        if output_df is not None:
            output_df.to_excel(writer, sheet_name=sheet_mult)
        if flags_df is not None:
            flags_df.to_excel(writer, sheet_name=sheet_flags)
        if attribution_df is not None:
            attribution_df.to_excel(writer, sheet_name=sheet_attr)

print(f'\nExported to: {EXCEL_OUTPUT}')
print("[PROGRESS] Export Complete")



All KMX RAGU Decomposition:


vintage,2025-11-30/2025-12-06,2025-12-07/2025-12-13,2025-12-14/2025-12-20,2025-12-21/2025-12-27,2025-12-28/2026-01-03,2026-01-04/2026-01-10,2026-01-11/2026-01-17,2026-01-18/2026-01-24,2026-01-25/2026-01-31,2026-02-01/2026-02-07,2026-02-08/2026-02-14,2026-02-15/2026-02-21,2026-02-22/2026-02-28,2026-03-01/2026-03-07,2026-03-08/2026-03-14,2026-03-15/2026-03-21,2026-03-22/2026-03-28,2026-03-29/2026-04-04,2026-04-05/2026-04-11,2026-04-12/2026-04-18,2026-04-19/2026-04-25
Contract Model Score,141.876946,142.139621,141.775187,141.957509,142.055772,141.774719,141.49681,141.899148,141.600977,142.161469,142.380543,142.071714,142.063357,142.386654,142.481352,142.596629,142.675067,143.04651,142.893235,142.635792,143.347729
Mountain3 Score,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gross Loss Adjustments,1.088939,1.22983,0.741891,0.777234,0.848139,0.521478,0.505559,0.645881,0.548297,0.586527,0.595631,-0.124317,-0.432309,-0.043651,-0.00457,0.182583,0.005056,0.178879,0.376676,0.425154,0.580074
Recovery Adjustments,1.305283,0.852443,1.494383,0.887119,1.341174,0.503374,0.67308,0.656552,0.605375,0.711589,0.26478,0.374905,0.088973,0.102626,0.656406,0.773252,0.862886,0.787263,0.758972,1.020917,1.555704
LTV Adjustments,-0.326046,1.131262,0.710486,0.90732,0.60155,0.545147,0.544337,0.024191,0.650091,0.395629,0.600958,0.828224,0.511399,0.346333,0.246315,0.539406,0.546457,0.54617,0.194025,0.417503,-1.045482
APR Adjustments,0.685136,0.774791,0.812633,0.988939,0.914396,0.09685,0.104489,0.173995,0.371827,0.411511,0.690206,0.659046,0.17732,0.53722,0.8133,0.840703,0.551128,0.703962,0.614129,0.573515,0.143813
RAGU Score,144.630258,146.127947,145.53458,145.518121,145.761032,143.441568,143.324275,143.399767,143.776567,144.266725,144.532118,143.809571,142.408739,143.329182,144.192804,144.932574,144.640594,145.262784,144.837036,145.072881,144.581838
LTV,1.610072,1.524077,1.547949,1.53669,1.554252,1.557535,1.557582,1.588531,1.551437,1.566306,1.554286,1.541194,1.559506,1.56922,1.575165,1.55787,1.557459,1.557475,1.578291,1.565017,1.656206



MTN 3.0 RAGU Decomposition:


vintage,2025-11-30/2025-12-06
Contract Model Score,139.536387
Mountain3 Score,0.0
Gross Loss Adjustments,1.367279
Recovery Adjustments,0.045978
LTV Adjustments,-0.979686
APR Adjustments,0.594396
RAGU Score,140.564354
LTV,1.651877



MTN 3.1 RAGU Decomposition:


...


MTN 3.2 RAGU Decomposition:


vintage,2026-01-25/2026-01-31,2026-02-01/2026-02-07,2026-02-08/2026-02-14,2026-02-15/2026-02-21,2026-02-22/2026-02-28,2026-03-01/2026-03-07,2026-03-08/2026-03-14,2026-03-15/2026-03-21,2026-03-22/2026-03-28,2026-03-29/2026-04-04,2026-04-05/2026-04-11,2026-04-12/2026-04-18,2026-04-19/2026-04-25
Contract Model Score,142.118567,142.297686,141.383512,141.863417,141.927108,142.053173,142.388615,142.360023,142.036108,142.437178,142.346796,142.367422,143.024535
Mountain3 Score,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gross Loss Adjustments,0.283252,0.612819,0.671006,-0.010106,-0.293954,0.189952,0.072738,0.129778,-0.022333,0.292106,0.484246,0.55448,0.658462
Recovery Adjustments,1.434241,0.825389,0.049675,0.176598,0.098091,-0.015211,0.449393,0.548123,1.070045,0.808906,0.8234,0.992817,1.964122
LTV Adjustments,-0.327164,0.572921,0.658483,0.895801,0.319903,-0.227754,-0.156724,0.160128,0.373182,0.331701,0.346624,0.370074,-0.79033
APR Adjustments,0.124431,0.041496,0.284926,0.322741,-0.058212,0.098373,0.575393,0.499423,0.20708,0.424118,0.438191,0.546934,0.108041
RAGU Score,143.633326,144.350312,143.047602,143.248452,141.992937,142.098534,143.329416,143.697476,143.664082,144.294009,144.439257,144.831727,144.96483
LTV,1.610142,1.555916,1.550951,1.537344,1.570787,1.603968,1.599585,1.580324,1.567632,1.570087,1.569203,1.567816,1.639545



MTN 4.1 RAGU Decomposition:


vintage,2025-12-14/2025-12-20,2025-12-21/2025-12-27,2025-12-28/2026-01-03,2026-01-04/2026-01-10,2026-01-11/2026-01-17,2026-01-18/2026-01-24,2026-01-25/2026-01-31,2026-02-01/2026-02-07,2026-02-08/2026-02-14,2026-02-15/2026-02-21,2026-02-22/2026-02-28,2026-03-01/2026-03-07,2026-03-08/2026-03-14,2026-03-15/2026-03-21,2026-03-22/2026-03-28,2026-03-29/2026-04-04,2026-04-05/2026-04-11,2026-04-12/2026-04-18,2026-04-19/2026-04-25
Contract Model Score,141.366824,143.015816,143.451321,142.266283,142.30266,142.672288,144.256069,143.888991,144.476556,143.55873,145.297113,145.106952,144.987739,145.002135,146.979511,146.487574,146.490621,144.706375,145.92277
Mountain3 Score,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gross Loss Adjustments,-0.938976,0.233329,-0.784207,-0.674341,-1.06448,-0.636178,-0.715642,-0.4848,-0.824067,-0.923423,-1.499922,-0.769102,-0.953048,-0.444409,-0.39169,-0.507266,-0.489527,-0.568986,-0.101401
Recovery Adjustments,3.023892,0.413572,1.459733,1.298141,1.536532,0.831213,-0.463144,1.170871,-0.46146,0.749835,0.038731,0.468361,1.609146,0.458366,0.555898,1.186116,0.839465,1.239843,-1.753913
LTV Adjustments,2.921514,0.684318,2.257514,1.04238,0.701439,0.387251,1.656275,1.33814,1.377146,1.646784,2.144365,1.865561,1.439133,2.141467,0.628655,1.114804,0.296649,0.787834,-3.06478
APR Adjustments,2.018163,1.013911,1.85007,0.441111,0.135537,0.249468,0.323397,0.576541,0.747007,1.057822,0.969595,1.130593,1.460847,1.441876,1.621648,1.19394,1.601882,0.77785,0.454807
RAGU Score,148.391418,145.360946,148.234431,144.373573,143.611689,143.504042,145.056955,146.489743,145.315182,146.089747,146.949882,147.802365,148.543816,148.599435,149.394023,149.47517,148.73909,146.942915,141.457482
LTV,1.430236,1.549458,1.463662,1.529058,1.54847,1.566801,1.495305,1.512609,1.510466,1.495816,1.469514,1.484136,1.507072,1.469664,1.552679,1.524997,1.572168,1.543505,1.801052



Exported to: new_kmx_models.xlsx
[PROGRESS] Export Complete


In [11]:
print("="*80)
print("  VERIFICATION: All KMX vs bareboned_ragu_new.ipynb")
print("="*80)

all_kmx_df = results_by_model.get('All KMX')
if all_kmx_df is not None and len(all_kmx_df) > 0:
    try:
        reference_df = pd.read_csv('all_df.csv')
        ref_kmx = reference_df[reference_df.lob == 'KMX'].copy() if 'lob' in reference_df.columns else pd.DataFrame()

        if len(ref_kmx) > 0:
            our_kmx = all_kmx_df.reset_index()
            our_kmx = our_kmx[our_kmx.lob == 'KMX'][['vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact', 'ragu_score', 'loss_multiplier']].copy()
            our_kmx = our_kmx.set_index('vintage')

            ref_cols = ['vintage', 'ms_original', 'ragu_score', 'loss_multiplier']
            ref_available = [c for c in ref_cols if c in ref_kmx.columns]
            ref_kmx_compare = ref_kmx[ref_available].copy()
            ref_kmx_compare = ref_kmx_compare.set_index('vintage')

            common_vintages = sorted(set(our_kmx.index) & set(ref_kmx_compare.index))

            if common_vintages:
                comparison = pd.DataFrame(index=common_vintages)
                for col in [c for c in ['ragu_score', 'loss_multiplier'] if c in ref_kmx_compare.columns]:
                    comparison[f'{col}_ours'] = our_kmx.loc[common_vintages, col].values
                    comparison[f'{col}_ref'] = ref_kmx_compare.loc[common_vintages, col].values
                    comparison[f'{col}_diff'] = comparison[f'{col}_ours'] - comparison[f'{col}_ref']

                print(f"\nComparing {len(common_vintages)} common vintages:")
                display(comparison)
            else:
                print("No common vintages found.")
        else:
            print("No KMX rows found in reference all_df.csv.")
    except FileNotFoundError:
        print("all_df.csv not found. Run bareboned_ragu_new.ipynb first to generate the reference file.")
    except Exception as e:
        print(f"Error loading reference data: {e}")
        print("\nManually compare the 'All KMX' sheet in new_kmx_models.xlsx")
        print("against the KMX rows in barebones_ragu.xlsx")
else:
    print("No 'All KMX' results to verify.")

# Summary table
print("\n" + "="*80)
print("  SUMMARY: RAGU Scores by MTN Model (most recent vintage)")
print("="*80)
summary_rows = []
baseline_ltv = BASELINES['KMX']['ltv']
for mtn_model_filter in MTN_MODELS + ['All KMX']:
    label = f'MTN {mtn_model_filter}' if mtn_model_filter != 'All KMX' else 'All KMX'
    model_df = results_by_model.get(mtn_model_filter)
    if model_df is not None and len(model_df) > 0:
        kmx_rows = model_df.reset_index()
        kmx_rows = kmx_rows[kmx_rows.lob == 'KMX']
        if len(kmx_rows) > 0:
            last_vintage = kmx_rows.vintage.max()
            last_row = kmx_rows[kmx_rows.vintage == last_vintage].iloc[0]
            summary_rows.append({
                'Model': label,
                'Latest Vintage': last_vintage,
                'Contract MS': last_row.get('ms_original', float('nan')),
                'Gross Loss Impact': last_row.get('gross_loss_impact', float('nan')),
                'Recovery Impact': last_row.get('recovery_impact', float('nan')),
                'LTV Impact': last_row.get('ltv_impact', float('nan')),
                'APR Impact': last_row.get('apr_impact', float('nan')),
                'RAGU Score': last_row.get('ragu_score', float('nan')),
                'LTV': last_row.get('ltv', float('nan')),
                'Loss Multiplier': last_row.get('loss_multiplier', float('nan')),
                'N Vintages': len(kmx_rows),
            })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).set_index('Model')
    display(summary_df)

    print(f"\nRAGU decomposition:")
    print(f"  baseline_ltv = {baseline_ltv}")
    print(f"  baseline_apr = {BASELINES['KMX']['apr']}")
    print(f"  RAGU Score = (unit_loss * recovery * baselined_recovery) + ltv_impact + apr_impact")
else:
    print("No results available.")

  VERIFICATION: All KMX vs bareboned_ragu_new.ipynb



Comparing 17 common vintages:


,ragu_score_ours,ragu_score_ref,ragu_score_diff
2025-12-28/2026-01-03,145.761032,145.830481,-0.069450
2026-01-04/2026-01-10,143.441568,143.488082,-0.046514
2026-01-11/2026-01-17,143.324275,143.401251,-0.076976
2026-01-18/2026-01-24,143.399767,143.490878,-0.091111
2026-01-25/2026-01-31,143.776567,143.852014,-0.075447
2026-02-01/2026-02-07,144.266725,144.343523,-0.076798
2026-02-08/2026-02-14,144.532118,144.601110,-0.068992
2026-02-15/2026-02-21,143.809571,143.884296,-0.074725
2026-02-22/2026-02-28,142.408739,142.493989,-0.085249
2026-03-01/2026-03-07,143.329182,143.399015,-0.069833



  SUMMARY: RAGU Scores by MTN Model (most recent vintage)


,Latest Vintage,Contract MS,Gross Loss Impact,Recovery Impact,LTV Impact,APR Impact,RAGU Score,LTV,Loss Multiplier,N Vintages
Model,,,,,,,,,,
MTN 3.0,2025-11-30/2025-12-06,139.536387,1.367279,0.045978,-0.979686,0.594396,140.564354,1.651877,0.945309,1
MTN 3.1,2026-04-05/2026-04-11,143.088368,0.625091,-0.158607,-1.703802,0.984015,142.835065,1.700799,0.974996,19
MTN 3.2,2026-04-19/2026-04-25,143.024535,0.658462,1.964122,-0.790330,0.108041,144.964830,1.639545,0.973662,13
MTN 4.1,2026-04-19/2026-04-25,145.922770,-0.101401,-1.753913,-3.064780,0.454807,141.457482,1.801052,1.004056,19
All KMX,2026-04-19/2026-04-25,143.347729,0.580074,1.555704,-1.045482,0.143813,144.581838,1.656206,0.976797,21



RAGU decomposition:
  baseline_ltv = 1.59
  baseline_apr = 0.25
  RAGU Score = (unit_loss * recovery * baselined_recovery) + ltv_impact + apr_impact
